In [1]:
from pycoingecko import CoinGeckoAPI
import pandas as pd
import plotly.graph_objects as go

# Initialize client
cg = CoinGeckoAPI()

# 1. Grab 30 days of historical data
bitcoin_data = cg.get_coin_market_chart_by_id(id='bitcoin', vs_currency='usd', days=30)

# 2. Load into DataFrame
data = pd.DataFrame(bitcoin_data['prices'], columns=['timestamp', 'prices'])

# Convert ms timestamp to Datetime
data['Date'] = pd.to_datetime(data['timestamp'], unit='ms')

# 3. Aggregate hourly data into Daily Open, High, Low, Close (OHLC)
candlestick_data = data.groupby(data['Date'].dt.date).agg({'prices': ['first', 'max', 'min', 'last']})

# 🔥 DEBUG FIX: Flatten the multi-layer columns so they are easy to reference
candlestick_data.columns = ['open', 'high', 'low', 'close']

# 4. Create the Interactive Candlestick Plot
fig = go.Figure(data=[go.Candlestick(
    x=candlestick_data.index,
    open=candlestick_data['open'],
    high=candlestick_data['high'],
    low=candlestick_data['low'],
    close=candlestick_data['close']
)])

# Style the Layout
fig.update_layout(
    title='Bitcoin Price Candlestick Chart (Last 30 Days)',
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    xaxis_rangeslider_visible=False,
    template="plotly_dark"  # Clean dark mode UI
)

fig.show()